# 信用卡欺诈预测基线

使用 Kaggle S6E11 官方 CSV，完成数据探索、LightGBM 五折验证和提交文件检查。更新于 2026-11-25。

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import RocCurveDisplay, roc_auc_score

from src.data import checksum, read_csv
from src.validate_submission import validate

data_dir = Path("data/raw")
run_dir = Path("artifacts/baseline/e01-lgb2")
files = [data_dir / name for name in ("train.csv", "test.csv", "sample_submission.csv")]
missing = [str(path) for path in files if not path.is_file()]
if missing:
    raise FileNotFoundError("请将 Kaggle 比赛数据解压到 data/raw。缺少 " + ", ".join(missing))

## 1. 探索数据

检查数据规模、缺失值和欺诈比例。用交易金额的十分位区间观察各组欺诈比例，横轴从低金额到高金额。

In [ ]:
train, test, sample = [read_csv(path) for path in files]
display(pd.DataFrame({
    "行数": [len(train), len(test)],
    "列数": [train.shape[1], test.shape[1]],
    "缺失值数": [train.isna().sum().sum(), test.isna().sum().sum()],
}, index=["训练集", "测试集"]))
print(f"训练集欺诈比例 {train['is_fraud'].mean():.2%}")

amount_group = pd.qcut(train["amount_usd"], 10, labels=False, duplicates="drop")
fraud_rate = train.groupby(amount_group)["is_fraud"].mean()
ax = (fraud_rate * 100).plot.bar(rot=0)
ax.set(xlabel="Transaction amount quantile group", ylabel="Fraud rate (%)")
plt.show()

## 2. 运行 LightGBM 五折训练

调用项目的正式基线。已有同名运行时，默认核对数据后读取其产物。要重新训练，把 `run_dir` 改为一个新名称，例如 `artifacts/baseline/e01-lgb2-notebook`。

In [ ]:
if not run_dir.exists():
    subprocess.run([
        sys.executable, "-m", "src.train", "fit", "e01-lgb2",
        "--phase", "baseline", "--data-dir", str(data_dir), "--name", run_dir.name,
    ], check=True)

run = json.loads((run_dir / "run.json").read_text())
if (
    not run["complete"]
    or run["experiment"] != "e01-lgb2"
    or run["file_sha256"] != {path.name: checksum(path) for path in files}
):
    raise ValueError("基线产物和当前数据不匹配。请使用新目录重新训练。")
print(f"基线产物目录 {run_dir}")

## 3. 查看 OOF AUC 和 ROC 曲线

每笔训练交易的分数来自留出它的那一折。这里同时查看合并 AUC 和各折 AUC。

In [ ]:
oof = read_csv(run_dir / "oof.csv")
if (
    oof["id"].duplicated().any()
    or len(oof) != len(train)
    or set(oof["fold"]) != set(range(5))
    or not oof.set_index("id")["is_fraud"].sort_index().astype(float).equals(
        train.set_index("id")["is_fraud"].sort_index().astype(float)
    )
):
    raise ValueError("OOF 数据与当前训练集或五折配置不匹配。请使用新目录重新训练。")

auc = roc_auc_score(oof["is_fraud"], oof["prediction"])
print(f"整体 OOF ROC AUC {auc:.6f}")
display(pd.Series({
    fold: roc_auc_score(rows["is_fraud"], rows["prediction"])
    for fold, rows in oof.groupby("fold")
}, name="ROC AUC").rename_axis("fold"))
RocCurveDisplay.from_predictions(oof["is_fraud"], oof["prediction"], name="LightGBM OOF")
plt.show()

## 4. 检查可提交的 CSV

完整五折训练会生成 `submission.csv`。检查列名、样例 ID 顺序和概率范围，打印文件哈希用于追溯。

In [ ]:
submission_path = run_dir / "submission.csv"
submission = read_csv(submission_path)
report = validate(submission, sample)
print(f"{submission_path} · {report['rows']:,} 行 · SHA-256 {checksum(submission_path)}")
display(submission.head())